In [42]:
pip install xgboost

  Using cached xgboost-3.0.2-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.0.2-py3-none-win_amd64.whl (150.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [44]:
# import libariers
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns


In [45]:
# loading dataset
df = pd.read_csv(r"C:\files from desktop\All projects\student performance prediction\new student data\student_performance_dataset.csv")

In [46]:
# basic structure of data
df.head()

,Student_ID,Gender,Study_Hours_per_Week,Attendance_Rate,Past_Exam_Scores,Parental_Education_Level,Internet_Access_at_Home,Extracurricular_Activities,Final_Exam_Score,Pass_Fail
0,S147,Male,31,68.267841,86,High School,Yes,Yes,63,Pass
1,S136,Male,16,78.222927,73,PhD,No,No,50,Fail
2,S209,Female,21,87.525096,74,PhD,Yes,No,55,Fail
3,S458,Female,27,92.076483,99,Bachelors,No,No,65,Pass
4,S078,Female,37,98.655517,63,Masters,No,Yes,70,Pass


In [47]:
df.shape

(708, 10)

In [48]:
df.dtypes

Student_ID                     object
Gender                         object
Study_Hours_per_Week            int64
Attendance_Rate               float64
Past_Exam_Scores                int64
Parental_Education_Level       object
Internet_Access_at_Home        object
Extracurricular_Activities     object
Final_Exam_Score                int64
Pass_Fail                      object
dtype: object

In [49]:
# checking null values
df.isnull().sum()

Student_ID                    0
Gender                        0
Study_Hours_per_Week          0
Attendance_Rate               0
Past_Exam_Scores              0
Parental_Education_Level      0
Internet_Access_at_Home       0
Extracurricular_Activities    0
Final_Exam_Score              0
Pass_Fail                     0
dtype: int64

In [50]:
# removing unwanted columns
df = df.drop(['Student_ID','Pass_Fail'], axis= 1)

In [51]:
df.head()

,Gender,Study_Hours_per_Week,Attendance_Rate,Past_Exam_Scores,Parental_Education_Level,Internet_Access_at_Home,Extracurricular_Activities,Final_Exam_Score
0,Male,31,68.267841,86,High School,Yes,Yes,63
1,Male,16,78.222927,73,PhD,No,No,50
2,Female,21,87.525096,74,PhD,Yes,No,55
3,Female,27,92.076483,99,Bachelors,No,No,65
4,Female,37,98.655517,63,Masters,No,Yes,70


In [52]:
print(df.columns)

Index(['Gender', 'Study_Hours_per_Week', 'Attendance_Rate', 'Past_Exam_Scores',
       'Parental_Education_Level', 'Internet_Access_at_Home',
       'Extracurricular_Activities', 'Final_Exam_Score'],
      dtype='object')


# handling categorical data for binary data
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder() 
df['Gender'] = le.fit_transform(df['Gender'])
df['Internet_Access_at_Home'] = le.fit_transform(df['Internet_Access_at_Home'])
df[ 'Extracurricular_Activities'] = le.fit_transform(df[ 'Extracurricular_Activities'])

print(df['Parental_Education_Level'].unique())

# handling categorical data for multi class data
from sklearn.preprocessing import OrdinalEncoder
edu_order = ['High School','PhD','Bachelors','Masters']
ord_enc = OrdinalEncoder(categories=[edu_order])
df['Parental_Education_Level'] = ord_enc.fit_transform(df[['Parental_Education_Level']])

# scaling numerical data
num_cols= ['Study_Hours_per_Week', 'Attendance_Rate', 'Past_Exam_Scores', 'Final_Exam_Score']

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [53]:
# print(df[num_cols].describe())

In [54]:
# split data

features = df.drop('Final_Exam_Score', axis=1)
prediction = df['Final_Exam_Score']

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(features,prediction,test_size=0.2, random_state=42)

In [55]:
from sklearn.preprocessing import OrdinalEncoder

# Education level encoding – define order
edu_order = ['High School', 'PhD', 'Bachelors', 'Masters']
ordinal_encoder = OrdinalEncoder(categories=[edu_order])

# Apply only on x_train, then transform x_test
x_train['Parental_Education_Level'] = ordinal_encoder.fit_transform(
    x_train[['Parental_Education_Level']]
)
x_test['Parental_Education_Level'] = ordinal_encoder.transform(
    x_test[['Parental_Education_Level']]
)


In [56]:
from sklearn.preprocessing import StandardScaler

# Optional: identify only numeric columns to scale (excluding encoded categories)
num_cols = ['Study_Hours_per_Week', 'Attendance_Rate', 'Past_Exam_Scores']

scaler = StandardScaler()
x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
x_test[num_cols] = scaler.transform(x_test[num_cols])


In [57]:
# Binary mapping
binary_map = {'Yes': 1, 'No': 0, 'Male': 0, 'Female': 1}

x_train['Internet_Access_at_Home'] = x_train['Internet_Access_at_Home'].map(binary_map)
x_test['Internet_Access_at_Home'] = x_test['Internet_Access_at_Home'].map(binary_map)

x_train['Gender'] = x_train['Gender'].map(binary_map)
x_test['Gender'] = x_test['Gender'].map(binary_map)

x_train['Extracurricular_Activities'] = x_train['Extracurricular_Activities'].map(binary_map)
x_test['Extracurricular_Activities'] = x_test['Extracurricular_Activities'].map(binary_map)


In [58]:
# build and train the model
from xgboost import XGBRegressor

model = XGBRegressor()
model.fit(x_train, y_train)

y_pred_test = model.predict(x_test)


In [62]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [63]:
# evaluating the model
mae= mean_absolute_error(y_test , y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test , y_pred_test))
r2 = r2_score(y_test , y_pred_test)
print(f"MAE: {mae}, RMSE: {rmse}, R2: {r2}")

MAE: 1.8118864784778004, RMSE: 3.0997804904397395, R2: 0.7741534059208977


In [64]:
# Evaluation
mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred_test)

print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")


MAE: 1.81
MSE: 9.61
RMSE: 3.10
R² Score: 0.77


In [65]:
import joblib

# Save the trained model
joblib.dump(model, 'xgb_model.pkl')
joblib.dump(scaler, 'scaler.pkl')  # e.g., StandardScaler
joblib.dump(ordinal_encoder, 'ordinal_encoder.pkl')  # e.g., OrdinalEncoder



['ordinal_encoder.pkl']